# SPU Stormwater EDA — v01

Starter notebook for exploring a representative sample of Seattle Public Utilities' stormwater asset inventory.

**Sample:** `~/Desktop/spu/sample/` — 12 CSV files, ~8,700 features across 9 asset classes.

Each row carries the original SPU attributes plus two geometry columns:

| Column | Purpose |
|---|---|
| `geometry_wkt` | Well-Known Text geometry (e.g. `POINT (-122.27 47.66)`). The universal portable format — usable in any mapping library after a single parse step. |
| `geometry_type` | `Point`, `LineString`, or `Polygon`. Useful for filtering or branching analysis. |

For non-spatial analysis you can ignore both. For mapping, parse `geometry_wkt` with `shapely.from_wkt` (and optionally wrap in `geopandas.GeoDataFrame`).

In [5]:
# import pandas as pd
# import numpy as np
# from pathlib import Path

## 1. Sample manifest

`_index.csv` is the cheat sheet — what's in the sample folder, asset class, geometry type, and how the sample compares to the full layer.

In [ ]:
index = pd.read_csv(SAMPLE / "_index.csv")
index

## 2. Load all sample layers

Stored as a `dict` keyed by the human-readable label, so you can grab one (`data["Storm Outfalls"]`) or iterate over all of them.

In [ ]:
data = {row["label"]: pd.read_csv(SAMPLE / row["file"]) for _, row in index.iterrows()}
print(f"Loaded {len(data)} layers, {sum(len(d) for d in data.values()):,} total features")
for label, df in data.items():
    print(f"  {label:<40} {len(df):>6,} rows  x  {len(df.columns):>3} cols")

## 3. Confirm geometry fields are present and populated

Sanity check: each layer should have a non-null `geometry_wkt` for every row, with a consistent `geometry_type`.

In [ ]:
rows = []
for label, df in data.items():
    rows.append({
        "layer": label,
        "rows": len(df),
        "attribute_cols": len(df.columns) - len(GEOMETRY_COLS),
        "geometry_type": df["geometry_type"].iloc[0] if len(df) else "",
        "wkt_populated": df["geometry_wkt"].notna().sum(),
        "sample_wkt": (df["geometry_wkt"].iloc[0][:55] + "...") if len(df) and pd.notna(df["geometry_wkt"].iloc[0]) else "",
    })
pd.DataFrame(rows)

## 4. Cross-sectional exploration

For analysis that doesn't need geometry — counts, distributions, correlations — drop the geometry columns for a cleaner view. Below are a couple of starter probes; replace with whatever you want to investigate.

### 4a. Storm Outfalls — ownership breakdown

In [ ]:
outfalls = data["Storm Outfalls"]
outfalls_attrs = outfalls.drop(columns=list(GEOMETRY_COLS))
print(f"Storm outfalls — {len(outfalls):,} rows, {len(outfalls_attrs.columns)} attribute columns")
outfalls["OUT_OWNER_NAME"].value_counts(dropna=False).head(10)

### 4b. Mainlines — pipe material distribution

In [ ]:
mains = data["Mainlines Permitted Use (sample)"]
# Find the material field — SPU's column name varies; let's look it up
material_col = next((c for c in mains.columns if "MTRL" in c.upper() and "TEXT" in c.upper()), None)
print(f"Using material column: {material_col}")
if material_col:
    mains[material_col].value_counts(dropna=False).head(10)

## 5. Mapping setup — parse WKT to real geometries

The standard pattern:

1. `shapely.from_wkt` converts the text strings into geometry objects.
2. (Optional) Wrap the resulting `DataFrame` in `geopandas.GeoDataFrame` for spatial joins, projection handling, and built-in plotting.
3. Hand to a mapping library — `folium` for interactive web maps, `contextily` for static basemaps, `kepler.gl` for fancier exploration, etc.

If the libraries aren't installed yet, in your terminal:

```bash
pip install shapely geopandas folium
```

All sample geometries are in **WGS84 (EPSG:4326)** — i.e., lon/lat in decimal degrees.

In [ ]:
try:
    from shapely import from_wkt
    import geopandas as gpd
    HAS_GIS = True

    def to_gdf(df: pd.DataFrame, crs: str = "EPSG:4326") -> "gpd.GeoDataFrame":
        """Parse `geometry_wkt` into shapely geometries and return a GeoDataFrame."""
        d = df.copy()
        d["geometry"] = d["geometry_wkt"].apply(from_wkt)
        return gpd.GeoDataFrame(d, geometry="geometry", crs=crs)

    gdf_outfalls = to_gdf(data["Storm Outfalls"])
    print(f"GeoDataFrame ready: {gdf_outfalls.shape}, CRS={gdf_outfalls.crs}")
    print(f"Bounds (minx, miny, maxx, maxy): {gdf_outfalls.total_bounds}")
except ImportError as e:
    HAS_GIS = False
    print(f"Mapping libraries not installed: {e}")
    print("Run:  pip install shapely geopandas folium")

## 5b. Derive uniform `lat` / `lon` on every layer

`geometry_wkt` is fully populated, but mapping libraries and downstream tooling often want a simple numeric `(lat, lon)` pair per row. The cell below adds `lat` and `lon` columns to every layer, derived from `geometry_wkt`:

- **Points**: lat / lon are the point coordinates directly.
- **Lines / Polygons**: lat / lon are the geometry's **centroid** — one representative point per feature.

After this runs, every DataFrame in `data[...]` carries `geometry_wkt`, `geometry_type`, `lat`, and `lon` populated 100% of the way down — a uniform, reliable pair regardless of geometry type.

In [ ]:
if HAS_GIS:
    def add_lat_lon(df: pd.DataFrame) -> pd.DataFrame:
        """Add `lat` and `lon` columns derived from `geometry_wkt`.
        Points -> point coords; lines/polygons -> centroid."""
        d = df.copy()
        geoms = d["geometry_wkt"].apply(lambda w: from_wkt(w) if pd.notna(w) else None)
        # .centroid on a Point returns the same point — uniform across geom types
        centroids = geoms.apply(lambda g: g.centroid if g is not None else None)
        d["lat"] = centroids.apply(lambda c: c.y if c is not None else None)
        d["lon"] = centroids.apply(lambda c: c.x if c is not None else None)
        return d

    data = {label: add_lat_lon(df) for label, df in data.items()}

    print(f"{'layer':<40} {'lat/lon%':>9}  {'lat range':<22} {'lon range':<22}")
    print("-" * 100)
    for label, df in data.items():
        ok = df[['lat', 'lon']].notna().all(axis=1).mean() * 100
        print(f"{label:<40} {ok:>8.1f}%  "
              f"({df['lat'].min():.4f}, {df['lat'].max():.4f})  "
              f"({df['lon'].min():.4f}, {df['lon'].max():.4f})")
else:
    print("Run the mapping-setup cell first (shapely is needed).")

## 6. Quick interactive map — Storm Outfalls

Folium drops a Leaflet map straight into the notebook. One marker per outfall, colored uniformly here; replace the styling with anything you want once you're exploring.

In [ ]:
if HAS_GIS:
    try:
        import folium
        df = data["Storm Outfalls"]  # now has lat / lon thanks to 5b
        center = [df["lat"].mean(), df["lon"].mean()]
        m = folium.Map(location=center, zoom_start=11, tiles="cartodbpositron")
        for _, row in df.iterrows():
            folium.CircleMarker(
                location=[row["lat"], row["lon"]],
                radius=2,
                color="#1F4E78",
                fill=True,
                fill_opacity=0.7,
                popup=folium.Popup(
                    f"Owner: {row.get('OUT_OWNER_NAME', '?')}<br>"
                    f"Lifecycle: {row.get('OUT_LIFECYCLE_STAT', '?')}"
                ),
            ).add_to(m)
        m
    except ImportError:
        print("folium not installed:  pip install folium")
else:
    print("Run the previous cells first (mapping libs needed).")

## 7. Where to go from here

- **Cross-sectional probes**: lifecycle status by asset class, install-date distributions, ownership splits between SPU vs private, NPDES coverage rates, dimensions vs material, etc.
- **Spatial probes**: outfalls per drainage basin, GSI density vs catch basin density, distance from each outfall to nearest mainline endpoint, geometric centroid mapping for polygon assets.
- **Network probes**: join mainlines to their endpoints (via `MNL_UPS_ENDPT_ID` / `MNL_DNS_ENDPT_ID`) to reconstruct the topology; trace upstream contributors to a given outfall.
- **Promotion to the full dataset**: once a probe is shaped well on the sample, swap the file path to `~/Desktop/spu/geojson/` (the full `.geojson.gz` files) and the same logic should work — just remember to use `geopandas.read_file()` directly on the gzipped GeoJSON instead of `pd.read_csv`.